In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif, mutual_info_classif

### Load Data Sources

In [ ]:
# Load features generated in 01_data_exploration (section 7.3)
processed_dir = Path('../..') / 'data' / 'processed'
parquet_path = processed_dir / 'timeseries_features.parquet'
csv_path = processed_dir / 'timeseries_features.csv'

if parquet_path.exists():
    df_features = pd.read_parquet(parquet_path)
elif csv_path.exists():
    df_features = pd.read_csv(csv_path)
else:
    raise FileNotFoundError('Feature table not found. Run section 7.3 in notebook 01 first.')

# Load patient demographics
base_path = Path('../..') / 'data' / 'raw' / 'pads-parkinsons-disease-smartwatch-dataset-1.0.0'
patients_path = base_path / 'patients'
questionnaire_path = base_path / 'questionnaire'

patient_files = sorted(list(patients_path.glob('*.json')))
questionnaire_files = sorted(list(questionnaire_path.glob('*.json')))

patient_rows = []
for fp in patient_files:
    with open(fp, 'r') as f:
        patient_rows.append(json.load(f))
df_patients = pd.DataFrame(patient_rows)

# Load questionnaire items (subject-level yes/no responses)
questionnaire_rows = []
for fp in questionnaire_files:
    with open(fp, 'r') as f:
        q = json.load(f)
    subject_id = q.get('subject_id')
    questionnaire_name = q.get('questionnaire_name')
    for item in q.get('item', []):
        questionnaire_rows.append({
            'subject_id': subject_id,
            'questionnaire_name': questionnaire_name,
            'link_id': item.get('link_id'),
            'answer': item.get('answer')
        })

df_questionnaire_items = pd.DataFrame(questionnaire_rows)
if len(df_questionnaire_items) > 0:
    df_questionnaire_items['answer_num'] = df_questionnaire_items['answer'].map({True: 1, False: 0})

print(f'df_features shape: {df_features.shape}')
print(f'df_patients shape: {df_patients.shape}')
print(f'df_questionnaire_items shape: {df_questionnaire_items.shape}')
print(df_features.head())
print(df_patients.head())
print(df_questionnaire_items.head())

df_features shape: (10318, 33)
df_patients shape: (469, 14)
df_questionnaire_items shape: (14070, 5)
   subject_id condition  record_name device_location  \
0           1   Healthy      Relaxed       LeftWrist   
1           1   Healthy      Relaxed      RightWrist   
2           1   Healthy  RelaxedTask       LeftWrist   
3           1   Healthy  RelaxedTask      RightWrist   
4           1   Healthy  StretchHold       LeftWrist   

                                   file_name  rows_expected  n_rows_raw  \
0       timeseries/001_Relaxed_LeftWrist.txt           2048        2048   
1      timeseries/001_Relaxed_RightWrist.txt           2048        2048   
2   timeseries/001_RelaxedTask_LeftWrist.txt           2048        2048   
3  timeseries/001_RelaxedTask_RightWrist.txt           2048        2048   
4   timeseries/001_StretchHold_LeftWrist.txt           1024        1024   

   n_rows_used  row_coverage  sampling_hz  ...  acc_mag_mean  acc_mag_std  \
0         1997           1.0   100

### Build Patient-Level Dataset

In [3]:
# 1) Build patient-level table from file-level sensor features
feature_exclude = {'rows_expected'}
feature_cols_file = [
    c for c in df_features.columns
    if c not in ['subject_id', 'condition', 'record_name', 'device_location', 'file_name']
    and c not in feature_exclude
    and np.issubdtype(df_features[c].dtype, np.number)
]

patient_agg = (
    df_features
    .groupby(['subject_id', 'condition'], as_index=False)[feature_cols_file]
    .agg(['mean', 'std'])
)

patient_agg.columns = [
    col if isinstance(col, str) else f'{col[0]}_{col[1]}'
    for col in patient_agg.columns
]
patient_agg = patient_agg.rename(columns={'subject_id_': 'subject_id', 'condition_': 'condition'})

def normalize_subject_id(series):
    s = series.astype(str).str.strip()
    s = s.str.replace(r'\.0$', '', regex=True)
    s = s.str.replace(r'^(patient_|subject_)', '', regex=True)
    s = s.str.extract(r'(\d+)', expand=False).fillna(s)
    s = s.apply(lambda x: x.zfill(3) if isinstance(x, str) and x.isdigit() and len(x) < 3 else x)
    return s

patient_agg['subject_id'] = normalize_subject_id(patient_agg['subject_id'])

# 2) Merge demographics (fixed columns known from dataset)
demo_cols = ['age', 'gender', 'height', 'weight']
df_demo = df_patients[['id'] + demo_cols].copy().rename(columns={'id': 'subject_id'})
df_demo['subject_id'] = normalize_subject_id(df_demo['subject_id'])
patient_df = patient_agg.merge(df_demo, on='subject_id', how='left')

# 3) Add questionnaire features (subject-level)
#    - one column per question
if len(df_questionnaire_items) > 0 and 'answer_num' in df_questionnaire_items.columns:
    q_work = df_questionnaire_items.copy()
    q_work['subject_id'] = normalize_subject_id(q_work['subject_id'])

    q_features = (
        q_work
        .pivot(index='subject_id', columns='link_id', values='answer_num')
        .add_prefix('q_')
        .reset_index()
    )

    patient_df['subject_id'] = normalize_subject_id(patient_df['subject_id'])
    patient_df = patient_df.merge(q_features, on='subject_id', how='left')

# 4) Define binary target y = PD vs non-PD
condition_norm = patient_df['condition'].astype(str).str.lower().str.strip()
y = (condition_norm == "parkinson's disease").astype(int)

if y.sum() == 0:
    y = condition_norm.str.contains('parkinson', regex=False).astype(int)
    y = np.where(condition_norm.str.contains('atypical|secondary|parkinsonism', regex=True), 0, y)
    y = pd.Series(y, index=patient_df.index)

patient_df['y_pd_binary'] = y

print(f'Patient-level rows: {len(patient_df)}')
print('Target distribution (0=non-PD, 1=PD):')
print(patient_df['y_pd_binary'].value_counts())

# Controlling the merged dataset quality
print('\nFirst 5 rows of merged patient_df:')
display(patient_df.head())

patient_missing = patient_df.isna().sum()
total_missing = int(patient_missing.sum())
cols_with_missing = int((patient_missing > 0).sum())
print('\nMerged patient_df missing report:')
print(f'Total missing values: {total_missing}')
print(f'Columns with at least 1 missing: {cols_with_missing}/{patient_df.shape[1]}')

top_missing_patient_df = patient_missing[patient_missing > 0].sort_values(ascending=False)
if len(top_missing_patient_df) > 0:
    print('Top columns by missing count:')
    print(top_missing_patient_df.head(15).to_string())
else:
    print('No missing values found in patient_df.')

Patient-level rows: 469
Target distribution (0=non-PD, 1=PD):
y_pd_binary
1    276
0    193
Name: count, dtype: int64

First 5 rows of merged patient_df:


,subject_id,condition,n_rows_raw_mean,n_rows_raw_std,n_rows_used_mean,n_rows_used_std,row_coverage_mean,row_coverage_std,sampling_hz_mean,sampling_hz_std,...,q_22,q_23,q_24,q_25,q_26,q_27,q_28,q_29,q_30,y_pd_binary
0,001,Healthy,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.696039,0.363960,...,0,0,0,0,0,0,0,0,0,0
1,002,Other Movement Disorders,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.693372,0.356011,...,1,1,0,1,0,1,0,1,0,0
2,003,Healthy,1303.272727,466.782521,1252.818182,466.858767,1.0,0.0,99.709051,0.379732,...,0,0,0,0,0,0,0,0,0,0
3,004,Parkinson's,1303.272727,466.782521,1252.818182,466.754308,1.0,0.0,99.687758,0.365762,...,1,1,0,0,1,1,1,0,0,1
4,005,Parkinson's,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.704164,0.378040,...,1,1,1,1,1,1,0,0,0,1



Merged patient_df missing report:
Total missing values: 0
Columns with at least 1 missing: 0/91
No missing values found in patient_df.


### Age-Aware Stratified Train/Test Split

In [16]:
# Keep all patients and reduce age imbalance with stratified splitting by class + age bins
if 'age' not in patient_df.columns:
    raise ValueError('Age column required for age-aware split')

work_df = patient_df.copy()
work_df['age'] = pd.to_numeric(work_df['age'], errors='coerce')
work_df = work_df.dropna(subset=['age', 'y_pd_binary']).copy()
work_df['y_pd_binary'] = work_df['y_pd_binary'].astype(int)

print(f'Total patients available: {len(work_df)}')
print('Class distribution (0=HC, 1=PD):')
print(work_df['y_pd_binary'].value_counts().sort_index())

# Build age bins and stratify using class + age bin
target_bins = 5
n_unique_ages = work_df['age'].nunique()
n_bins = min(target_bins, n_unique_ages)
if n_bins < 2:
    raise ValueError('Not enough age variability to create age bins for stratification')

work_df['age_bin'] = pd.qcut(work_df['age'], q=n_bins, duplicates='drop')
work_df['strata'] = work_df['y_pd_binary'].astype(str) + '__' + work_df['age_bin'].astype(str)

strata_counts = work_df['strata'].value_counts()
rare_strata = strata_counts[strata_counts < 2]

if len(rare_strata) > 0:
    print('Warning: some class+age strata have <2 samples. Falling back to class-only stratification.')
    stratify_labels = work_df['y_pd_binary']
else:
    stratify_labels = work_df['strata']

train_indices, test_indices = train_test_split(
    work_df.index,
    test_size=0.20,
    random_state=42,
    stratify=stratify_labels
)

# Split data
X_all = work_df.drop(columns=['subject_id', 'condition', 'y_pd_binary', 'age_bin', 'strata'], errors='ignore')
X_all = X_all.select_dtypes(include=[np.number])
y_all = work_df['y_pd_binary']

X_train = X_all.loc[train_indices].reset_index(drop=True)
X_test = X_all.loc[test_indices].reset_index(drop=True)
y_train = y_all.loc[train_indices].reset_index(drop=True)
y_test = y_all.loc[test_indices].reset_index(drop=True)

# Diagnostics: check that age and class stay balanced across splits
train_age = work_df.loc[train_indices, 'age']
test_age = work_df.loc[test_indices, 'age']

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Train PD: {int(y_train.sum())}, HC: {int(len(y_train) - y_train.sum())}')
print(f'Test PD: {int(y_test.sum())}, HC: {int(len(y_test) - y_test.sum())}')
print(f'Train age mean +- std: {train_age.mean():.2f} +- {train_age.std():.2f}')
print(f'Test age mean +- std: {test_age.mean():.2f} +- {test_age.std():.2f}')
print(f"Overall age mean +- std: {work_df['age'].mean():.2f} +- {work_df['age'].std():.2f}")

Total patients available: 469
Class distribution (0=HC, 1=PD):
y_pd_binary
0    193
1    276
Name: count, dtype: int64
X_train: (375, 87) | X_test: (94, 87)
Train PD: 221, HC: 154
Test PD: 55, HC: 39
Train age mean +- std: 64.16 +- 10.60
Test age mean +- std: 64.57 +- 11.13
Overall age mean +- std: 64.24 +- 10.70


### Objective Feature Selection (No Demographics)

In [17]:
# Remove demographic columns from feature selection
demo_cols_to_remove = ['age', 'weight', 'height']  # gender is categorical, already excluded
X_train_no_demo = X_train.drop(columns=[c for c in demo_cols_to_remove if c in X_train.columns])
X_test_no_demo = X_test.drop(columns=[c for c in demo_cols_to_remove if c in X_test.columns])

print(f'Features after demo removal: {X_train_no_demo.shape[1]} (removed {len(demo_cols_to_remove)} demo cols)')

# Define setups (sensor-only, questionnaire-only, all)
all_cols = X_train_no_demo.columns.tolist()
q_cols = [c for c in all_cols if c.startswith('q_')]
sensor_cols = [c for c in all_cols if not c.startswith('q_')]

setups = {
    'A_sensor_only': sensor_cols,
    'B_questionnaire_only': q_cols,
    'C_all_modalities': all_cols
}

def run_objective_feature_selection(X_train_in, X_test_in, y_train_in, top_k=25):
    # Missing filter
    train_missing_ratio = X_train_in.isna().mean()
    keep_missing = train_missing_ratio[train_missing_ratio <= 0.20].index.tolist()
    X_train_w = X_train_in[keep_missing].fillna(X_train_in[keep_missing].median())
    X_test_w = X_test_in[keep_missing].fillna(X_train_in[keep_missing].median())

    # Correlation filter
    corr_matrix = X_train_w.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
    X_train_f = X_train_w.drop(columns=to_drop_corr)
    X_test_f = X_test_w.drop(columns=to_drop_corr)

    # ANOVA + Mutual Info on clean features
    f_vals, p_vals = f_classif(X_train_f, y_train_in)
    mi_vals = mutual_info_classif(X_train_f, y_train_in, random_state=42)

    fs_df = pd.DataFrame({
        'feature': X_train_f.columns,
        'anova_f': f_vals,
        'anova_p': p_vals,
        'mutual_info': mi_vals
    })

    fs_df['rank_anova'] = fs_df['anova_p'].rank(ascending=True, method='average')
    fs_df['rank_mi'] = fs_df['mutual_info'].rank(ascending=False, method='average')
    fs_df['rank_combined'] = (fs_df['rank_anova'] + fs_df['rank_mi']) / 2
    fs_df = fs_df.sort_values('rank_combined', ascending=True).reset_index(drop=True)

    # Select top-k
    selected = fs_df.head(min(top_k, len(fs_df)))['feature'].tolist()
    X_train_sel = X_train_f[selected]
    X_test_sel = X_test_f[selected]

    info = {
        'n_input_features': len(X_train_in.columns),
        'n_after_missing': len(X_train_w.columns),
        'n_after_corr': len(X_train_f.columns),
        'n_selected': len(selected)
    }

    return X_train_sel, X_test_sel, fs_df, selected, info

Features after demo removal: 84 (removed 3 demo cols)


In [21]:
# Run selection for each setup
# Setup A and B: global top-25 within their modality pool (unchanged)
# Setup C: stratified budget — top-12 sensors + top-13 questionnaires = 25 total
setup_results = {}
setup_summary_rows = []

for setup_name, cols in setups.items():
    if setup_name == 'C_all_modalities':
        # Split columns by modality
        sensor_cols_c = [c for c in cols if not c.startswith('q_')]
        q_cols_c = [c for c in cols if c.startswith('q_')]

        # Independent selection per modality with fixed budget
        Xtr_s, Xte_s, fs_s, feat_s, info_s = run_objective_feature_selection(
            X_train_no_demo[sensor_cols_c].copy(),
            X_test_no_demo[sensor_cols_c].copy(),
            y_train, top_k=12
        )
        Xtr_q, Xte_q, fs_q, feat_q, info_q = run_objective_feature_selection(
            X_train_no_demo[q_cols_c].copy(),
            X_test_no_demo[q_cols_c].copy(),
            y_train, top_k=13
        )

        Xtr_sel = pd.concat([Xtr_s.reset_index(drop=True), Xtr_q.reset_index(drop=True)], axis=1)
        Xte_sel = pd.concat([Xte_s.reset_index(drop=True), Xte_q.reset_index(drop=True)], axis=1)
        feat_list = feat_s + feat_q
        fs_table = pd.concat([fs_s, fs_q], ignore_index=True)
        info = {
            'n_input_features': info_s['n_input_features'] + info_q['n_input_features'],
            'n_after_missing': info_s['n_after_missing'] + info_q['n_after_missing'],
            'n_after_corr': info_s['n_after_corr'] + info_q['n_after_corr'],
            'n_selected': len(feat_list)
        }
        print(f"{setup_name}: top-12 sensors ({len(feat_s)}) + top-13 questionnaires ({len(feat_q)}) = {len(feat_list)} features")
    else:
        Xtr = X_train_no_demo[cols].copy()
        Xte = X_test_no_demo[cols].copy()
        Xtr_sel, Xte_sel, fs_table, feat_list, info = run_objective_feature_selection(
            Xtr, Xte, y_train, top_k=25
        )

    setup_results[setup_name] = {
        'X_train_selected': Xtr_sel,
        'X_test_selected': Xte_sel,
        'feature_selection_table': fs_table,
        'final_feature_list': feat_list,
        'info': info
    }

    setup_summary_rows.append({
        'setup': setup_name,
        **info
    })

setup_summary_df = pd.DataFrame(setup_summary_rows).sort_values('setup').reset_index(drop=True)
print('\nObjective feature selection summary (age-aware stratified split, no demo):')
print(setup_summary_df)

for setup_name in sorted(setup_results.keys()):
    r = setup_results[setup_name]
    feats = r['final_feature_list']
    n_sensor = sum(1 for f in feats if not f.startswith('q_'))
    n_quest = sum(1 for f in feats if f.startswith('q_'))
    print(f"\n{setup_name}: train/test -> "
          f"{r['X_train_selected'].shape} / "
          f"{r['X_test_selected'].shape} "
          f"({n_sensor} sensor, {n_quest} questionnaire)")


c:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\bhi-env\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [4 5 7 8] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\bhi-env\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\bhi-env\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [4 5 7 8] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\bhi-env\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


C_all_modalities: top-12 sensors (12) + top-13 questionnaires (13) = 25 features

Objective feature selection summary (age-aware stratified split, no demo):
                  setup  n_input_features  n_after_missing  n_after_corr  \
0         A_sensor_only                54               54            36   
1  B_questionnaire_only                30               30            30   
2      C_all_modalities                84               84            66   

   n_selected  
0          25  
1          25  
2          25  

A_sensor_only: train/test -> (375, 25) / (94, 25) (25 sensor, 0 questionnaire)

B_questionnaire_only: train/test -> (375, 25) / (94, 25) (0 sensor, 25 questionnaire)

C_all_modalities: train/test -> (375, 25) / (94, 25) (12 sensor, 13 questionnaire)


### Save Selected Datasets

In [22]:
# Save selected datasets
final_dir = Path('../..') / 'data' / 'final' / 'no_demo_objective'
final_dir.mkdir(parents=True, exist_ok=True)

for setup_name, results in setup_results.items():
    # Save train/test datasets
    results['X_train_selected'].to_csv(final_dir / f'X_train_{setup_name}_selected_no_demo_objective.csv', index=False)
    results['X_test_selected'].to_csv(final_dir / f'X_test_{setup_name}_selected_no_demo_objective.csv', index=False)

    # Save feature selection table
    results['feature_selection_table'].to_csv(final_dir / f'feature_selection_table_{setup_name}_no_demo_objective.csv', index=False)

    # Save feature list
    pd.DataFrame({'feature': results['final_feature_list']}).to_csv(
        final_dir / f'final_feature_list_{setup_name}_no_demo_objective.csv', index=False
    )

# Save labels (same for all setups)
y_train.to_csv(final_dir / 'y_train_no_demo_objective.csv', index=False)
y_test.to_csv(final_dir / 'y_test_no_demo_objective.csv', index=False)

print(f'\nSaved all datasets to: {final_dir}')

# Summary
print('\n=== SUMMARY ===')
print('Created objective feature selection with:')
print('- Age-aware stratified train/test split (class + age bins)')
print('- Demographics excluded from feature selection')
print('- Top 25 features per setup based on ANOVA + Mutual Info')
print('- Datasets saved in data/final/no_demo_objective/')


Saved all datasets to: ..\..\data\final\no_demo_objective

=== SUMMARY ===
Created objective feature selection with:
- Age-aware stratified train/test split (class + age bins)
- Demographics excluded from feature selection
- Top 25 features per setup based on ANOVA + Mutual Info
- Datasets saved in data/final/no_demo_objective/
